In [4]:
#!/usr/bin/env python3
"""
Redrob Hackathon — Intelligent Candidate Ranker v2
===================================================
Usage:
    python rank.py --candidates candidates.jsonl --out submission.csv
    python rank.py --candidates candidates.jsonl.gz --out submission.csv

Architecture: Two-pass hybrid ranker
  Pass 1: Hard disqualifier filter  (~0.5s for 100K)
  Pass 2: Multi-signal scoring with 6 components + behavioral multiplier

Scoring formula:
  final = (
      0.32 * skill_score          # JD-weighted, evidence-gated
    + 0.26 * career_score         # product trajectory, role relevance
    + 0.10 * location_score       # Pune/Noida preferred
    + 0.05 * education_score      # pedigree (minor signal)
  ) * behavioral_multiplier        # availability × engagement gate
  - anti_pattern_penalty          # consulting-only, keyword-stuffer, honeypot

Constraints met: CPU-only, no network, <5 min / 100K, <2GB RAM
"""

import argparse
import csv
import gzip
import json
import math
import re
from datetime import date, datetime
from pathlib import Path
from collections import defaultdict

# ── Reference date ─────────────────────────────────────────────────────────────
REF_DATE = date(2026, 6, 27)

# ── JD skill taxonomy ──────────────────────────────────────────────────────────
# Each entry: canonical_name -> (jd_weight, [alias_tokens...])
# alias_tokens are word-boundary matched against skill name + career text
SKILL_TAXONOMY = [
    # Core retrieval / ranking — MUST HAVE
    ("embeddings",              3.5, ["embed", "embedding", "embeddings", "dense retrieval", "bi-encoder"]),
    ("sentence-transformers",   3.2, ["sentence-transformer", "sentence_transformer", "sbert", "bge", "e5 model", "gte", "all-minilm"]),
    ("vector database",         3.2, ["vector db", "vector store", "vectordb", "vector index", "ann index", "approximate nearest"]),
    ("faiss",                   3.0, ["faiss"]),
    ("pinecone",                3.0, ["pinecone"]),
    ("weaviate",                3.0, ["weaviate"]),
    ("qdrant",                  3.0, ["qdrant"]),
    ("milvus",                  2.8, ["milvus"]),
    ("elasticsearch",           2.8, ["elasticsearch", "opensearch", "elastic search"]),
    ("hybrid search",           3.0, ["hybrid search", "hybrid retrieval", "bm25", "sparse dense", "sparse+dense"]),
    ("ranking system",          3.5, ["ranking", "ranker", "rerank", "re-rank", "reranking", "learning to rank", "ltr"]),
    ("information retrieval",   3.2, ["information retrieval", "ir system", "search engine", "search system", "search infrastructure"]),
    ("rag",                     3.0, ["rag", "retrieval augmented", "retrieval-augmented"]),
    ("nlp",                     2.8, ["nlp", "natural language processing", "text processing", "language model"]),
    ("evaluation framework",    3.0, ["ndcg", "mrr", "map@", "precision@", "recall@", "a/b test", "ab test", "offline eval", "online eval", "ranking eval", "search eval"]),
    ("recommendation system",   2.8, ["recommend", "recommender", "collaborative filtering", "content-based filtering"]),
    ("python",                  2.2, ["python"]),
    ("applied ml",              2.5, ["applied ml", "applied machine learning", "production ml", "ml engineer", "ml system"]),
    # Nice-to-have
    ("llm fine-tuning",         2.0, ["lora", "qlora", "peft", "fine-tun", "finetuning", "fine_tun", "instruct tuning", "sft"]),
    ("llm",                     1.8, ["llm", "large language model", "gpt", "claude", "llama", "mistral", "palm", "gemini"]),
    ("transformers",            1.5, ["transformer", "attention mechanism", "bert", "roberta", "t5", "huggingface", "hf model"]),
    ("pytorch",                 1.4, ["pytorch", "torch"]),
    ("distributed systems",     1.3, ["distributed", "spark", "kafka", "flink", "ray"]),
    ("inference optimization",  1.3, ["inference optim", "model serving", "triton", "onnx", "tensorrt", "quantization", "distillation"]),
    ("xgboost",                 1.2, ["xgboost", "lightgbm", "catboost", "gradient boost"]),
    # Downweighted — JD explicitly warns about
    ("langchain",               0.2, ["langchain", "lang_chain"]),
    ("openai api",              0.3, ["openai api", "chatgpt api", "gpt api wrapper"]),
]

# Build flat lookup: token -> (canonical, weight, is_nice)
_TOKEN_MAP: dict[str, tuple[str, float]] = {}
SKILL_WEIGHTS: dict[str, float] = {}
for canonical, weight, aliases in SKILL_TAXONOMY:
    SKILL_WEIGHTS[canonical] = weight
    for token in aliases:
        _TOKEN_MAP[token.lower()] = (canonical, weight)

MAX_SKILL_SCORE = sum(w for _, w, _ in SKILL_TAXONOMY if w >= 2.0)  # must-have ceiling

# ── Consulting firm detector ───────────────────────────────────────────────────
CONSULTING_TOKENS = {
    "tcs", "tata consultancy", "infosys", "wipro", "accenture", "cognizant",
    "capgemini", "hcl technologies", "hcltech", "tech mahindra", "mphasis",
    "hexaware", "ltimindtree", "mindtree", "persistent systems", "niit tech",
    "cyient", "zensar", "birlasoft", "mastech", "syntel", "igate",
    "deloitte consulting", "kpmg consulting", "pwc consulting",
}

# ── Wrong-domain skills (CV/speech/robotics per JD) ──────────────────────────
WRONG_DOMAIN_TOKENS = {
    "computer vision", "image classification", "object detection",
    "image segmentation", "face recognition", "facial recognition",
    "speech recognition", "text-to-speech", "tts system", "asr",
    "robotics", "lidar", "slam", "point cloud", "action recognition",
    "video understanding", "scene understanding",
}

# ── Location scores ───────────────────────────────────────────────────────────
# Ordered: longest match first to avoid "delhi" matching "new delhi"
LOCATION_TABLE = [
    ("new delhi",   0.85), ("delhi ncr",  0.85), ("ncr",        0.82),
    ("noida",       1.00), ("pune",       1.00), ("gurugram",   0.85),
    ("gurgaon",     0.85), ("faridabad",  0.80), ("greater noida", 0.88),
    ("delhi",       0.83), ("mumbai",     0.80), ("navi mumbai", 0.78),
    ("hyderabad",   0.78), ("secunderabad", 0.76), ("bangalore", 0.75),
    ("bengaluru",   0.75), ("chennai",    0.65), ("kolkata",    0.60),
    ("ahmedabad",   0.58), ("jaipur",     0.55), ("kochi",      0.55),
    ("india",       0.45),
]

JD_SAL_MIN, JD_SAL_MAX = 18.0, 65.0


# ════════════════════════════════════════════════════════════════════════════════
# PASS 1: HARD DISQUALIFIERS
# ════════════════════════════════════════════════════════════════════════════════

def hard_disqualify(c: dict) -> str | None:
    """
    Returns a disqualification reason string, or None if candidate passes.
    Per JD: pure research, consulting-only, CV/speech only, no AI exp,
    outside India without relocation, honeypot patterns.
    """
    profile  = c.get("profile", {})
    career   = c.get("career_history", [])
    skills   = c.get("skills", [])
    signals  = c.get("redrob_signals", {})
    edu      = c.get("education", [])
    yoe      = profile.get("years_of_experience", 0)

    # ── 1. Honeypot: impossible experience vs education ────────────────────
    if edu:
        earliest_grad = min((e.get("end_year", 9999) for e in edu), default=9999)
        if earliest_grad < 9999:
            max_possible = REF_DATE.year - earliest_grad + 1
            if yoe > max_possible + 2:
                return f"honeypot:yoe_{yoe}_impossible_grad_{earliest_grad}"

    # ── 2. Honeypot: career duration > claimed experience by large margin ──
    if career:
        total_months = sum(r.get("duration_months", 0) for r in career)
        if total_months > (yoe + 4) * 12:
            return f"honeypot:duration_sum_{total_months}mo_vs_yoe_{yoe}"

    # ── 3. Honeypot: all signals maxed out ────────────────────────────────
    rr  = signals.get("recruiter_response_rate", 0)
    icr = signals.get("interview_completion_rate", 0)
    oar = signals.get("offer_acceptance_rate", -1)
    pc  = signals.get("profile_completeness_score", 0)
    github = signals.get("github_activity_score", -1)
    if rr >= 0.99 and icr >= 0.99 and oar >= 0.99 and pc >= 99.9 and github >= 99:
        return "honeypot:all_signals_perfect"

    # ── 4. Honeypot: skill count wildly exceeds career evidence ──────────
    if len(skills) > 50 and yoe < 4:
        return f"honeypot:skill_count_{len(skills)}_vs_yoe_{yoe}"

    # ── 5. Honeypot: future dates in career history ───────────────────────
    for role in career:
        sd = role.get("start_date", "")
        if sd and sd > "2026-07":
            return f"honeypot:future_start_date_{sd}"

    # ── 6. Clearly wrong role with no AI/ML evidence ──────────────────────
    # Tier A: always disqualify regardless of skills
    hard_wrong_titles = [
        "marketing manager", "sales manager", "hr manager", "human resources",
        "graphic designer", "ux designer", "product designer",
        "financial analyst", "accountant", "operations manager",
        "supply chain", "logistics manager", "legal counsel",
    ]
    # Tier B: wrong engineering domain — disqualify only if zero AI skill evidence
    domain_wrong_titles = [
        "civil engineer", "mechanical engineer", "electrical engineer",
        "chemical engineer", "hardware engineer", "frontend engineer",
        "front-end engineer", "front end engineer", "ios developer",
        "android developer", "mobile developer", "java developer",
        ".net developer", "php developer", "ruby developer",
        "ui developer", "web developer", "full stack developer",
        "fullstack developer", "devops engineer", "sre engineer",
        "network engineer", "security engineer",
    ]
    current_title = profile.get("current_title", "").lower()

    if any(t in current_title for t in hard_wrong_titles):
        return f"wrong_role:{current_title}"

    if any(t in current_title for t in domain_wrong_titles):
        # Allow through if they have strong AI skill evidence in career
        career_blob = " ".join(
            (r.get("title", "") + " " + r.get("description", "")).lower()
            for r in career
        )
        skill_names_lower = " ".join(s.get("name", "").lower() for s in skills)
        ai_signals = [
            "embed", "retriev", "rank", "vector", "nlp", "machine learning",
            "recommender", "search engine", "faiss", "pinecone", "qdrant",
            "weaviate", "milvus", "transformer", "bert", "llm", "rag",
        ]
        ai_hits_career = sum(1 for kw in ai_signals if kw in career_blob)
        ai_hits_skills = sum(1 for kw in ai_signals if kw in skill_names_lower)
        if ai_hits_career + ai_hits_skills < 2:
            return f"wrong_role:{current_title} (no AI evidence)"

    # ── 8. Completely non-technical career history ────────────────────────
    if career:
        tech_role_kws = [
            "engineer", "developer", "scientist", "architect", "analyst",
            "ml", "data", "research", "technical", "software", "platform",
        ]
        non_tech_role_kws = [
            "marketing", "sales", "customer support", "customer service",
            "account manager", "business development", "content writer",
            "hr ", "recruiter", "finance", "legal", "supply chain",
        ]
        tech_months  = 0
        total_career = sum(r.get("duration_months", 0) for r in career)
        for role in career:
            title_l = role.get("title", "").lower()
            dur = role.get("duration_months", 0)
            if any(kw in title_l for kw in tech_role_kws):
                tech_months += dur
        if total_career > 0 and tech_months / total_career < 0.10:
            return f"non_technical_career:tech_months={tech_months}/{total_career}"

    # ── 9. Ghost candidate: very long inactive + not open to work ─────────
    last_active_str = signals.get("last_active_date", "2020-01-01")
    open_to_work    = signals.get("open_to_work_flag", False)
    try:
        last_active  = datetime.strptime(last_active_str, "%Y-%m-%d").date()
        days_dark    = (REF_DATE - last_active).days
    except Exception:
        days_dark = 999
    if days_dark > 180 and not open_to_work:
        return f"ghost_candidate:inactive_{days_dark}d_not_open"



    # ── 10. Zero years experience ─────────────────────────────────────────
    if yoe < 1:
        return "insufficient_experience:yoe<1"

    return None  # passes hard filter


# ════════════════════════════════════════════════════════════════════════════════
# PASS 2: SCORING
# ════════════════════════════════════════════════════════════════════════════════

# Pre-compiled regex for fast token matching in text blobs
_SKILL_REGEXES: dict[str, re.Pattern] = {}
for token in _TOKEN_MAP:
    escaped = re.escape(token)
    try:
        _SKILL_REGEXES[token] = re.compile(rf'\b{escaped}\b', re.IGNORECASE)
    except re.error:
        _SKILL_REGEXES[token] = re.compile(re.escape(token), re.IGNORECASE)


def _build_career_blob(c: dict) -> str:
    """Concatenate all career text for fast regex search."""
    parts = []
    for role in c.get("career_history", []):
        parts.append(role.get("title", ""))
        parts.append(role.get("description", ""))
        parts.append(role.get("company", ""))
    return " ".join(parts).lower()


def _build_profile_blob(c: dict) -> str:
    """Concatenate headline + summary."""
    p = c.get("profile", {})
    return (p.get("headline", "") + " " + p.get("summary", "")).lower()


def score_skills(c: dict) -> tuple[float, list[str]]:
    """
    Returns (score 0–1, list of matched skill names).
    Three evidence tiers:
      Tier A: Listed skill with proficiency + duration + endorsements
      Tier B: Mentioned in career description (evidence of practice)
      Tier C: Mentioned in profile headline/summary only (weakest)
    """
    skills      = c.get("skills", [])
    sig         = c.get("redrob_signals", {})
    assessments = sig.get("skill_assessment_scores", {})
    career_blob = _build_career_blob(c)
    profile_blob = _build_profile_blob(c)

    # Build skill name → skill object lookup (normalised)
    skill_lookup: dict[str, dict] = {}
    for s in skills:
        name = s.get("name", "").lower().strip()
        skill_lookup[name] = s

    matched_weight = 0.0
    matched_names  = []
    seen_canonical = set()

    for token, (canonical, jd_weight) in _TOKEN_MAP.items():
        if canonical in seen_canonical:
            continue

        # --- Find best evidence tier ---
        best_s    = None
        evidence  = 0  # 0=none, 1=profile_only, 2=career_desc, 3=listed_skill

        # Tier A: explicit skill entry
        for sname, sobj in skill_lookup.items():
            if token in sname or sname in token:
                if evidence < 3:
                    best_s   = sobj
                    evidence = 3
                    break

        # Tier B: career description
        if evidence < 2:
            pattern = _SKILL_REGEXES.get(token)
            if pattern and pattern.search(career_blob):
                evidence = 2
                # Estimate duration from matching role
                for role in c.get("career_history", []):
                    desc = (role.get("title","") + " " + role.get("description","")).lower()
                    if pattern.search(desc):
                        dur = role.get("duration_months", 6)
                        best_s = {"proficiency": "intermediate", "endorsements": 1,
                                  "duration_months": dur}
                        break

        # Tier C: headline/summary only
        if evidence < 1:
            pattern = _SKILL_REGEXES.get(token)
            if pattern and pattern.search(profile_blob):
                evidence = 1
                best_s = {"proficiency": "beginner", "endorsements": 0, "duration_months": 3}

        if evidence == 0 or best_s is None:
            continue

        seen_canonical.add(canonical)
        matched_names.append(canonical)

        # Evidence multiplier
        ev_mult = {3: 1.0, 2: 0.65, 1: 0.35}[evidence]

        # Proficiency multiplier
        prof_mult = {"beginner": 0.35, "intermediate": 0.65,
                     "advanced": 0.88, "expert": 1.0}.get(
            best_s.get("proficiency", "intermediate"), 0.65)

        # Endorsement trust (log-scaled, cap 100)
        ends = best_s.get("endorsements", 0)
        end_mult = min(1.0, math.log1p(ends) / math.log1p(80))

        # Duration evidence (cap 48 months = 4 years of practice)
        dur = best_s.get("duration_months", 6)
        dur_mult = min(1.0, dur / 36)

        # Platform assessment bonus (±0.2 swing)
        assess_bonus = 0.0
        for ak, av in assessments.items():
            if token in ak.lower() or canonical.lower() in ak.lower():
                assess_bonus = (av - 50) / 250
                break

        contribution = jd_weight * ev_mult * (
            0.35 * prof_mult
            + 0.25 * end_mult
            + 0.25 * dur_mult
            + 0.15 * (1.0 + assess_bonus)
        )
        matched_weight += contribution

    normalised = min(1.0, matched_weight / (MAX_SKILL_SCORE * 0.45))
    return normalised, matched_names


def score_career(c: dict) -> tuple[float, dict]:
    """
    Career trajectory quality score + metadata dict.
    """
    career  = c.get("career_history", [])
    profile = c.get("profile", {})
    yoe     = profile.get("years_of_experience", 0)

    meta = {
        "product_ratio": 0.0,
        "relevance_ratio": 0.0,
        "is_consulting_only": False,
        "avg_tenure_months": 0,
        "company_switches_6yr": 0,
    }

    if not career:
        return 0.0, meta

    consulting_months = 0
    product_months    = 0
    relevant_months   = 0
    durations         = []
    total_months      = max(1, sum(r.get("duration_months", 0) for r in career))

    # Recent companies (last 6 years) for title-chaser detection
    recent_cos = set()

    # ML relevance keywords (expanded — catches "built a ranker", "shipped search")
    rel_kws = [
        "embed", "retriev", "rank", "search", "vector", r"\bnlp\b", "language model",
        "recommend", "neural", "transformer", "bert", "llm", "machine learning",
        r"\bml\b", "pipeline", "a/b", "evaluation", "inference", "model serv",
        "similarity", "semantic", "index", "query",
    ]
    rel_pattern = re.compile("|".join(rel_kws), re.IGNORECASE)

    for role in career:
        co    = role.get("company", "").lower()
        dur   = role.get("duration_months", 0)
        desc  = role.get("description", "").lower()
        title = role.get("title", "").lower()
        sd    = role.get("start_date", "")

        is_consulting = any(cf in co for cf in CONSULTING_TOKENS)
        if is_consulting:
            consulting_months += dur
        else:
            product_months += dur

        if rel_pattern.search(desc) or rel_pattern.search(title):
            relevant_months += dur

        durations.append(dur)

        # Title-chaser: company switches in last 6 years
        if sd >= "2020-01-01":
            recent_cos.add(role.get("company", ""))

    meta["product_ratio"]    = product_months / total_months
    meta["relevance_ratio"]  = relevant_months / total_months
    meta["is_consulting_only"] = (consulting_months == total_months) and len(career) >= 2
    meta["avg_tenure_months"] = (sum(durations) / len(durations)) if durations else 0
    meta["company_switches_6yr"] = len(recent_cos)

    s = 0.0

    # Current title fit
    senior_title_kws = [
        "ml engineer", "machine learning engineer", "ai engineer",
        "applied scientist", "nlp engineer", "search engineer",
        "research engineer", "platform engineer", "data scientist",
        "software engineer", "senior engineer", "staff engineer",
        "principal engineer",
    ]
    cur_title = profile.get("current_title", "").lower()
    if any(kw in cur_title for kw in senior_title_kws):
        s += 0.18
    elif any(kw in cur_title for kw in ["engineer", "developer", "scientist", "analyst"]):
        s += 0.09

    # Product company trajectory (big signal per JD)
    s += 0.28 * meta["product_ratio"]

    # Relevance of actual work
    s += 0.28 * meta["relevance_ratio"]

    # Tenure stability (target ≥ 18mo average — penalises title-chasers)
    tenure_score = min(1.0, meta["avg_tenure_months"] / 24)
    s += 0.10 * tenure_score

    # Years-of-experience fit band: 5–9yr ideal
    if 5 <= yoe <= 9:
        exp_s = 1.0
    elif yoe > 9:
        exp_s = max(0.55, 1.0 - (yoe - 9) * 0.04)
    elif yoe >= 3:
        exp_s = 0.55 + (yoe - 3) / 4 * 0.45
    else:
        exp_s = yoe / 3 * 0.55
    s += 0.16 * exp_s

    return min(1.0, s), meta


def score_behavioral(c: dict) -> float:
    """
    Availability × engagement gate from 23 Redrob signals.
    Returns a multiplier (0.05 – 1.50).
    A ghost candidate maxes out at ~0.35 regardless of technical score.
    """
    sig = c.get("redrob_signals", {})

    # ── Recency (most important signal) ──────────────────────────────────
    last_active_str = sig.get("last_active_date", "2020-01-01")
    try:
        last_active = datetime.strptime(last_active_str, "%Y-%m-%d").date()
        days_dark   = max(0, (REF_DATE - last_active).days)
    except Exception:
        days_dark = 365
    recency = math.exp(-days_dark / 45)   # half-life 45 days

    # ── Explicit availability ─────────────────────────────────────────────
    open_flag = 1.0 if sig.get("open_to_work_flag") else 0.20

    # ── Responsiveness ────────────────────────────────────────────────────
    rr         = float(sig.get("recruiter_response_rate", 0.5))
    avg_rt_hrs = float(sig.get("avg_response_time_hours", 72))
    speed      = math.exp(-avg_rt_hrs / 36)   # half-life 36 hours

    # ── Interview commitment ──────────────────────────────────────────────
    icr = float(sig.get("interview_completion_rate", 0.5))

    # ── Offer track record ────────────────────────────────────────────────
    oar = sig.get("offer_acceptance_rate", -1)
    offer_s = 0.55 if oar == -1 else max(0.0, float(oar))

    # ── Notice period ─────────────────────────────────────────────────────
    notice = sig.get("notice_period_days", 90)
    if notice <= 14:
        notice_s = 1.0
    elif notice <= 30:
        notice_s = 0.90
    elif notice <= 60:
        notice_s = 0.70
    elif notice <= 90:
        notice_s = 0.50
    else:
        notice_s = max(0.10, 0.50 - (notice - 90) / 180)

    # ── Platform engagement ───────────────────────────────────────────────
    profile_views = sig.get("profile_views_received_30d", 0)
    view_s  = min(1.0, math.log1p(profile_views) / math.log1p(30))
    saved   = sig.get("saved_by_recruiters_30d", 0)
    saved_s = min(1.0, math.log1p(saved) / math.log1p(8))
    apps    = sig.get("applications_submitted_30d", 0)
    apps_s  = min(1.0, math.log1p(apps) / math.log1p(10))

    # ── Technical signals ─────────────────────────────────────────────────
    github = sig.get("github_activity_score", -1)
    github_s = (float(github) / 100) if github >= 0 else 0.25

    # ── Profile quality ────────────────────────────────────────────────────
    completeness = float(sig.get("profile_completeness_score", 50)) / 100

    # ── Trust ─────────────────────────────────────────────────────────────
    verified = (
        int(bool(sig.get("verified_email")))
        + int(bool(sig.get("verified_phone")))
        + int(bool(sig.get("linkedin_connected")))
    ) / 3.0

    # ── Weighted combination ───────────────────────────────────────────────
    behavioral = (
        0.22 * recency
        + 0.20 * open_flag
        + 0.14 * rr
        + 0.10 * notice_s
        + 0.08 * icr
        + 0.07 * github_s
        + 0.05 * speed
        + 0.04 * completeness
        + 0.03 * offer_s
        + 0.03 * verified
        + 0.02 * view_s
        + 0.01 * saved_s
        + 0.01 * apps_s
    )

    # Convert to multiplier: 0.0 behav → 0.08 mult; 1.0 → 1.40 mult
    # Ghost candidates are effectively capped at 0.08 * (1 + 0.3) ≈ 0.10
    multiplier = 0.08 + 1.32 * behavioral
    return round(min(1.50, max(0.05, multiplier)), 4)


def score_location(c: dict) -> float:
    profile = c.get("profile", {})
    sig     = c.get("redrob_signals", {})

    loc     = (profile.get("location", "") + " " + profile.get("country", "")).lower()
    willing = bool(sig.get("willing_to_relocate", False))

    # Country check first
    is_india = ("india" in loc) or profile.get("country", "").lower() in ("in", "india", "")
    if not is_india:
        return 0.15 + (0.12 if willing else 0.0)

    # City match (ordered longest first)
    for city, base_score in LOCATION_TABLE:
        if city in loc:
            reloc_bonus = 0.0  # already in the city
            return base_score

    # India but city not recognised
    return 0.42 + (0.15 if willing else 0.0)


def score_education(c: dict) -> float:
    edu = c.get("education", [])
    if not edu:
        return 0.40

    tier_map = {"tier_1": 1.0, "tier_2": 0.80, "tier_3": 0.60,
                "tier_4": 0.40, "unknown": 0.45}
    cs_fields = {"computer science", "engineering", "machine learning",
                 "data science", "mathematics", "statistics", "ai", "it"}

    best = 0.0
    for e in edu:
        tier = tier_map.get(e.get("tier", "unknown"), 0.45)
        field = e.get("field_of_study", "").lower()
        field_bonus = 0.12 if any(f in field for f in cs_fields) else 0.0
        best = max(best, min(1.0, tier + field_bonus))
    return best


def anti_pattern_score(c: dict, career_meta: dict) -> float:
    """
    Returns deduction 0–0.35.
    """
    penalty  = 0.0
    profile  = c.get("profile", {})
    skills   = c.get("skills", [])
    sig      = c.get("redrob_signals", {})
    yoe      = profile.get("years_of_experience", 0)

    # Consulting-only career
    if career_meta.get("is_consulting_only"):
        penalty += 0.28

    # CV/speech domain with no NLP/IR
    skill_names = " ".join(s.get("name", "").lower() for s in skills)
    wrong_hits = sum(1 for t in WRONG_DOMAIN_TOKENS if t in skill_names)
    nlp_hits   = sum(1 for t in ["nlp", "retriev", "search", "embeddings", "ranking"]
                     if t in skill_names)
    if wrong_hits >= 2 and nlp_hits == 0:
        penalty += 0.22

    # Title-chaser: > 4 distinct companies in last 6 years
    switches = career_meta.get("company_switches_6yr", 0)
    if switches > 4:
        penalty += min(0.18, (switches - 4) * 0.04)

    # Ghost keyword stuffer: expert skills with zero evidence
    ghost_expert = sum(
        1 for s in skills
        if s.get("proficiency") in ("advanced", "expert")
        and s.get("duration_months", 0) == 0
        and s.get("endorsements", 0) == 0
    )
    if ghost_expert > 4:
        penalty += min(0.12, (ghost_expert - 4) * 0.02)

    # LangChain tutorial + very low experience flag (per JD)
    has_langchain = any("langchain" in s.get("name", "").lower() for s in skills)
    if has_langchain and yoe < 2:
        penalty += 0.08

    return min(0.35, penalty)


def salary_gate(c: dict) -> float:
    """Multiplier 0.35–1.0 based on salary range overlap with JD."""
    sig = c.get("redrob_signals", {})
    sal = sig.get("expected_salary_range_inr_lpa", {})
    s_min = float(sal.get("min", 0))
    s_max = float(sal.get("max", 999))

    # Malformed: min > max — treat as unknown, mild penalty
    if s_max < s_min:
        return 0.75

    if s_max <= 0:
        return 0.90  # unknown — don't penalise

    overlap_lo = max(s_min, JD_SAL_MIN)
    overlap_hi = min(s_max, JD_SAL_MAX)

    if overlap_hi < overlap_lo:
        # No overlap
        gap = (s_min - JD_SAL_MAX) if s_min > JD_SAL_MAX else (JD_SAL_MIN - s_max)
        return max(0.35, 1.0 - gap / JD_SAL_MAX)

    jd_range = JD_SAL_MAX - JD_SAL_MIN
    overlap   = overlap_hi - overlap_lo
    return 0.65 + 0.35 * min(1.0, overlap / jd_range)


# ════════════════════════════════════════════════════════════════════════════════
# COMPOSITE + REASONING
# ════════════════════════════════════════════════════════════════════════════════

def score_candidate(c: dict) -> tuple[float, str]:
    """Returns (score 0–1, reasoning string)."""

    # ── Hard filter ────────────────────────────────────────────────────────
    reason = hard_disqualify(c)
    if reason:
        snippet = reason[:80]
        return 0.0, f"Excluded: {snippet}"

    # ── Component scores ──────────────────────────────────────────────────
    skill_s,   matched_skills = score_skills(c)
    career_s,  career_meta    = score_career(c)
    behav_mult                = score_behavioral(c)
    loc_s                     = score_location(c)
    edu_s                     = score_education(c)
    anti_pen                  = anti_pattern_score(c, career_meta)
    sal_mult                  = salary_gate(c)

    # ── Composite ─────────────────────────────────────────────────────────
    # Base: weighted sum of structural signals
    base = (
        0.40 * skill_s
        + 0.33 * career_s
        + 0.17 * loc_s
        + 0.10 * edu_s
    )
    # Apply behavioral multiplier (availability gate)
    gated = base * behav_mult
    # Salary gate (soft)
    gated = gated * sal_mult
    # Anti-pattern deduction (after gating so penalty isn't amplified)
    final = max(0.0, min(1.0, gated - anti_pen))

    # ── Minimum viable skill floor ────────────────────────────────────────
    # If no must-have skills matched at all, cap at 0.15 to prevent
    # behaviorally-active but technically-irrelevant candidates ranking high
    must_have_canonicals = {can for can, w, _ in SKILL_TAXONOMY if w >= 2.5}
    matched_must = [s for s in matched_skills if s in must_have_canonicals]
    if len(matched_must) == 0:
        final = min(final, 0.12)
    elif len(matched_must) == 1:
        final = min(final, 0.35)

    # ── Reasoning ─────────────────────────────────────────────────────────
    reasoning = _build_reasoning(
        c, skill_s, career_s, behav_mult, loc_s,
        anti_pen, matched_skills, career_meta
    )
    return round(final, 6), reasoning


def _build_reasoning(c, skill_s, career_s, behav_mult, loc_s,
                     anti_pen, matched_skills, career_meta) -> str:
    profile = c.get("profile", {})
    sig     = c.get("redrob_signals", {})

    yoe    = profile.get("years_of_experience", 0)
    title  = profile.get("current_title", "")
    loc    = profile.get("location", "")
    notice = sig.get("notice_period_days", 90)
    rr     = sig.get("recruiter_response_rate", 0)
    github = sig.get("github_activity_score", -1)
    last_a = sig.get("last_active_date", "unknown")
    sal    = sig.get("expected_salary_range_inr_lpa", {})

    parts = []

    # Experience + title
    parts.append(f"{yoe:.0f}yr exp, {title}")

    # Top matched skills
    top_skills = [s for s in matched_skills if s in SKILL_WEIGHTS and SKILL_WEIGHTS[s] >= 2.5][:4]
    if top_skills:
        parts.append("skills: " + ", ".join(top_skills))

    # Career quality
    if career_meta.get("is_consulting_only"):
        parts.append("consulting-only career (penalty)")
    elif career_meta.get("product_ratio", 0) >= 0.7:
        parts.append("strong product-company trajectory")
    elif career_meta.get("product_ratio", 0) >= 0.4:
        parts.append("mixed product/consulting background")

    # Location
    parts.append(f"{loc or 'location unknown'}")

    # Logistics
    sal_str = ""
    if sal:
        sal_str = f" {sal.get('min',0):.0f}–{sal.get('max',0):.0f} LPA"
    parts.append(f"notice {notice}d; response {int(rr*100)}%{sal_str}")

    # GitHub
    if github >= 0:
        parts.append(f"GitHub {github:.0f}/100")

    parts.append(f"last active {last_a}")

    return "; ".join(parts) + "."


# ════════════════════════════════════════════════════════════════════════════════
# I/O
# ════════════════════════════════════════════════════════════════════════════════

def load_candidates(path: str) -> list[dict]:
    p = Path(path)
    opener = gzip.open(p, "rt", encoding="utf-8") if p.suffix == ".gz" else open(p, encoding="utf-8")
    candidates = []
    with opener as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    candidates.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return candidates


def rank_candidates(candidates: list) -> list[dict]:
    scored = []
    excluded = 0
    for c in candidates:
        try:
            score, reasoning = score_candidate(c)
            scored.append({"candidate_id": c["candidate_id"], "score": score, "reasoning": reasoning})
            if score == 0.0:
                excluded += 1
        except Exception as e:
            scored.append({"candidate_id": c.get("candidate_id", "?"), "score": 0.0,
                           "reasoning": f"Scoring error: {e}"})

    # Sort: descending score, tie-break candidate_id ascending (deterministic)
    scored.sort(key=lambda x: (-x["score"], x["candidate_id"]))
    print(f"  Excluded (disqualified/honeypot): {excluded:,}")
    return scored


def write_submission(scored: list, out_path: str):
    top100 = scored[:100]
    # Enforce non-increasing scores (guard against float rounding)
    for i in range(1, len(top100)):
        if top100[i]["score"] > top100[i-1]["score"]:
            top100[i]["score"] = top100[i-1]["score"]

    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["candidate_id", "rank", "score", "reasoning"])
        w.writeheader()
        for rank, row in enumerate(top100, 1):
            w.writerow({
                "candidate_id": row["candidate_id"],
                "rank":         rank,
                "score":        round(row["score"], 6),
                "reasoning":    row["reasoning"],
            })
    print(f"Written {len(top100)} rows → {out_path}")


# ════════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ════════════════════════════════════════════════════════════════════════════════

def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--candidates", required=True)
    ap.add_argument("--out", default="submission.csv")
    # Pass argv explicitly to avoid Colab/IPython kernel args poisoning sys.argv
    args = ap.parse_args(argv)

    print(f"Loading candidates from {args.candidates}...")
    candidates = load_candidates(args.candidates)
    print(f"Loaded {len(candidates):,} candidates")

    print("Scoring candidates...")
    scored = rank_candidates(candidates)

    n = len(scored)
    active = sum(1 for r in scored[:100] if r["score"] > 0)
    honeypots_in_top100 = 100 - active
    print(f"Honeypots / disqualified in top 100: {honeypots_in_top100}")
    print(f"Score range top 100: {scored[0]['score']:.4f} → {scored[min(99,n-1)]['score']:.4f}")

    write_submission(scored, args.out)
    print(f"Done. Validate with: python validate_submission.py {args.out}")


# ── Notebook / Colab entry point ─────────────────────────────────────────────
# In a Colab cell, skip main() entirely and call run() directly:
#
#   from rank import run
#   run(candidates_path="/content/candidates.jsonl", out_path="submission.csv")
#
def run(candidates_path: str, out_path: str = "submission.csv"):
    """Call this from a Colab/Jupyter cell instead of main()."""
    print(f"Loading candidates from {candidates_path}...")
    candidates = load_candidates(candidates_path)
    print(f"Loaded {len(candidates):,} candidates")

    print("Scoring candidates...")
    scored = rank_candidates(candidates)

    n = len(scored)
    active = sum(1 for r in scored[:100] if r["score"] > 0)
    print(f"Honeypots / disqualified in top 100: {100 - active}")
    print(f"Score range top 100: {scored[0]['score']:.4f} → {scored[min(99, n-1)]['score']:.4f}")

    write_submission(scored, out_path)
    print(f"Done. Validate with: python validate_submission.py {out_path}")


if __name__ == "__main__":
    # Use the run function for Colab execution, providing a sample candidate file.
    # The candidates.jsonl file is available in the kernel files.
    run(candidates_path="candidates.jsonl")

Loading candidates from candidates.jsonl...
Loaded 2,164 candidates
Scoring candidates...
  Excluded (disqualified/honeypot): 1,565
Honeypots / disqualified in top 100: 0
Score range top 100: 0.8943 → 0.2557
Written 100 rows → submission.csv
Done. Validate with: python validate_submission.py submission.csv


In [6]:
#!/usr/bin/env python3
"""
Validate submission CSV per challenge rules (sections 2–3).
Row 1 = header. Rows 2–101 = exactly 100 data rows. CSV only.
"""

import csv
import re
import sys
from pathlib import Path

REQUIRED_HEADER = ["candidate_id", "rank", "score", "reasoning"]
CANDIDATE_ID_PATTERN = re.compile(r"^CAND_[0-9]{7}$")
DATA_ROW_START = 2
EXPECTED_DATA_ROWS = 100


def validate_submission(csv_path):
    errors = []
    path = Path(csv_path)

    if path.suffix.lower() != ".csv":
        errors.append("Filename must use a .csv extension.")
    elif not path.stem:
        errors.append("Filename must be your registered participant ID (e.g. team_xxx.csv).")

    try:
        with open(path, "r", encoding="utf-8", newline="") as f:
            reader = csv.reader(f)

            try:
                header = next(reader)
            except StopIteration:
                errors.append("Row 1 must be the header row; file is empty.")
                return errors

            # Row 1: column names and their order come from this line only
            if header != REQUIRED_HEADER:
                errors.append(
                    "Row 1 (header) must be exactly:\n"
                    f"  {','.join(REQUIRED_HEADER)}\n"
                    f"Found:\n"
                    f"  {','.join(header)}"
                )

            data_rows = []
            for row in reader:
                if any(cell.strip() for cell in row):
                    data_rows.append(row)

    except UnicodeDecodeError:
        errors.append("File must be UTF-8 encoded.")
        return errors
    except OSError as e:
        errors.append(f"Cannot read file: {e}")
        return errors

    n = len(data_rows)
    if n != EXPECTED_DATA_ROWS:
        errors.append(
            f"After the header (row 1), there must be exactly {EXPECTED_DATA_ROWS} "
            f"data rows (rows {DATA_ROW_START}–{DATA_ROW_START + EXPECTED_DATA_ROWS - 1}); "
            f"found {n}."
        )

    seen_ids = set()
    seen_ranks = set()
    by_rank = []

    for i, cells in enumerate(data_rows):
        row_num = DATA_ROW_START + i

        if len(cells) != len(REQUIRED_HEADER):
            errors.append(
                f"Row {row_num}: expected {len(REQUIRED_HEADER)} columns "
                f"({','.join(REQUIRED_HEADER)}), got {len(cells)}."
            )
            continue

        row = dict(zip(REQUIRED_HEADER, cells))
        cid = row["candidate_id"].strip()
        rank_s = row["rank"].strip()
        score_s = row["score"].strip()

        if not cid:
            errors.append(f"Row {row_num}: candidate_id is required.")
        elif not CANDIDATE_ID_PATTERN.match(cid):
            errors.append(
                f"Row {row_num}: candidate_id must be CAND_XXXXXXX (7 digits)."
            )
        elif cid in seen_ids:
            errors.append(f"Row {row_num}: duplicate candidate_id '{cid}'.")
        else:
            seen_ids.add(cid)

        try:
            rank = int(rank_s)
            if str(rank) != rank_s:
                raise ValueError
            if not 1 <= rank <= 100:
                errors.append(f"Row {row_num}: rank must be between 1 and 100.")
            elif rank in seen_ranks:
                errors.append(f"Row {row_num}: duplicate rank {rank}.")
            else:
                seen_ranks.add(rank)
        except ValueError:
            errors.append(f"Row {row_num}: rank must be an integer (1–100).")
            rank = None

        try:
            score = float(score_s)
        except ValueError:
            errors.append(f"Row {row_num}: score must be a float.")
            score = None

        if rank is not None and score is not None and cid:
            by_rank.append((rank, score, cid))

    missing = set(range(1, 101)) - seen_ranks
    if missing:
        errors.append(
            f"Each rank 1–100 must appear exactly once; missing: {sorted(missing)}"
        )

    by_rank.sort(key=lambda x: x[0])

    for i in range(len(by_rank) - 1):
        r1, s1, _ = by_rank[i]
        r2, s2, _ = by_rank[i + 1]
        if s1 < s2:
            errors.append(
                f"score must be non-increasing by rank: "
                f"rank {r1} ({s1}) < rank {r2} ({s2})."
            )

    for i in range(len(by_rank) - 1):
        r1, s1, c1 = by_rank[i]
        r2, s2, c2 = by_rank[i + 1]
        if s1 == s2 and c1 > c2:
            errors.append(
                f"Equal scores at ranks {r1} and {r2}: "
                f"tie-break requires candidate_id ascending "
                f"({c1!r} > {c2!r})."
            )

    return errors


def main():
    if len(sys.argv) != 2:
        print("Usage: python validate_submission.py <participant_id>.csv")
        sys.exit(1)

    errors = validate_submission(sys.argv[1])
    if errors:
        print(f"Validation failed ({len(errors)} issue(s)):\n")
        for e in errors:
            print(f"- {e}")
        sys.exit(1)

    print("Submission is valid.")


def run(csv_path: str = "submission.csv"):
    """Call this from a Colab/Jupyter cell instead of main()."""
    errors = validate_submission(csv_path)
    if errors:
        print(f"Validation failed ({len(errors)} issue(s)):\n")
        for e in errors:
            print(f"- {e}")
        # In Colab, we generally don't want to sys.exit() unless critical
    else:
        print("Submission is valid.")


if __name__ == "__main__":
    # In a Colab environment, sys.argv might not have the expected arguments.
    # Call the run function with a default value for direct execution in Colab.
    run()


Submission is valid.
